# Scanpy QC — Ancestral (Cell Ranger filtered)
***
Input: `mergedh5_nova_anc_CRfiltered.h5ad`  
Cell calls: already determined by Cell Ranger 7.1.0 `filtered_feature_bc_matrix.h5`  
This notebook computes QC metrics and applies a minimal global filter floor.  
No additional cell calling is performed.

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sc.settings.verbosity = 1

# ── CONFIGURE ─────────────────────────────────────────────────────
WD = Path("/project2/araman/annisa")          # adjust if needed
INPUT  = WD / "mergedh5_nova_anc_CRfiltered.h5ad"
OUTPUT = WD / "anc_processed.h5ad"

# Global filter thresholds (applied after CR cell calling)
MIN_GENES = 200    # hard floor — removes near-empty barcodes CR may have included
MIN_COUNTS = 300   # UMI floor consistent with MIN_GENES
MAX_GENES  = 4000  # doublet guard — yeast transcriptome ~6000 genes total
# ──────────────────────────────────────────────────────────────────

In [ ]:
# ── LOAD ──────────────────────────────────────────────────────────
adata = sc.read_h5ad(INPUT)
print(adata)
print(f"\nConditions: {adata.obs['condition'].nunique()}")
print(adata.obs['condition'].value_counts())

In [ ]:
# ── COMPUTE QC METRICS ────────────────────────────────────────────
# Mitochondrial genes — yeast mt genes are prefixed Q0 or have 'MT-' style names
# In S. cerevisiae w303, mitochondrial genes in the SGD annotation start with 'Q0'
adata.var["mt"] = adata.var_names.str.startswith("Q0")
print(f"Mitochondrial genes detected: {adata.var['mt'].sum()}")

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=[50, 100, 200, 500],
    log1p=True,
    inplace=True,
)

print("\nQC metrics computed. obs columns:")
print([c for c in adata.obs.columns if any(
    c.startswith(p) for p in
    ["n_genes", "total_counts", "pct_counts", "log1p"]
)])

In [ ]:
# ── PRE-FILTER SUMMARY ────────────────────────────────────────────
print(f"Cells before filtering: {adata.n_obs}")
print(f"\nn_genes_by_counts:")
print(adata.obs["n_genes_by_counts"].describe().round(1))
print(f"\ntotal_counts:")
print(adata.obs["total_counts"].describe().round(1))

In [ ]:
# ── VISUALIZE PRE-FILTER DISTRIBUTIONS ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(adata.obs["n_genes_by_counts"], bins=100, color="#4C72B0", edgecolor="none")
axes[0].axvline(MIN_GENES, color="red", linestyle="--", linewidth=1, label=f"min={MIN_GENES}")
axes[0].axvline(MAX_GENES, color="orange", linestyle="--", linewidth=1, label=f"max={MAX_GENES}")
axes[0].set_xlabel("Genes per cell")
axes[0].set_ylabel("Cells")
axes[0].legend(fontsize=8)
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].hist(adata.obs["total_counts"], bins=100, color="#4C72B0", edgecolor="none")
axes[1].axvline(MIN_COUNTS, color="red", linestyle="--", linewidth=1, label=f"min={MIN_COUNTS}")
axes[1].set_xlabel("UMI counts per cell")
axes[1].set_ylabel("Cells")
axes[1].legend(fontsize=8)
axes[1].spines[["top", "right"]].set_visible(False)

axes[2].scatter(
    adata.obs["total_counts"],
    adata.obs["n_genes_by_counts"],
    s=1, alpha=0.2, color="#4C72B0", rasterized=True
)
axes[2].axvline(MIN_COUNTS, color="red", linestyle="--", linewidth=0.8)
axes[2].axhline(MIN_GENES, color="red", linestyle="--", linewidth=0.8)
axes[2].axhline(MAX_GENES, color="orange", linestyle="--", linewidth=0.8)
axes[2].set_xlabel("UMI counts per cell")
axes[2].set_ylabel("Genes per cell")
axes[2].spines[["top", "right"]].set_visible(False)

plt.suptitle("Ancestral — pre-filter QC distributions", fontsize=11)
plt.tight_layout()
plt.savefig(WD / "anc_qc_prefilter.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── VIOLIN PLOTS PER CONDITION (pre-filter) ───────────────────────
sc.pl.violin(
    adata,
    keys=["n_genes_by_counts", "total_counts"],
    groupby="condition",
    rotation=90,
    stripplot=False,
    show=False,
)
plt.suptitle("Per-condition QC — pre-filter", fontsize=10, y=1.01)
plt.savefig(WD / "anc_qc_violin_prefilter.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── APPLY GLOBAL FILTERS ──────────────────────────────────────────
# Cell Ranger filtered output is the cell-calling foundation.
# These thresholds remove residual low-quality barcodes only.

n_before = adata.n_obs

sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_cells(adata, min_counts=MIN_COUNTS)
sc.pp.filter_cells(adata, max_genes=MAX_GENES)

# Gene filter: keep genes expressed in at least 1 cell
sc.pp.filter_genes(adata, min_cells=1)

n_after = adata.n_obs
print(f"Cells before filter: {n_before}")
print(f"Cells after filter:  {n_after}")
print(f"Cells removed:       {n_before - n_after}  ({100*(n_before-n_after)/n_before:.1f}%)")
print(f"\nGenes retained: {adata.n_vars}")

In [ ]:
# ── POST-FILTER SUMMARY PER CONDITION ────────────────────────────
post_filter = (
    adata.obs
    .groupby("condition")[["n_genes_by_counts", "total_counts"]]
    .agg(["count", "median"])
    .round(1)
)
post_filter.columns = ["n_cells", "median_genes", "_n", "median_umi"]
post_filter = post_filter[["n_cells", "median_genes", "median_umi"]]
print("Post-filter cells and metrics per condition:")
print(post_filter.to_string())

In [ ]:
# ── CHECK: all conditions still meet paper thresholds? ────────────
below_1000 = post_filter[post_filter["n_cells"] < 1000]
below_500g = post_filter[post_filter["median_genes"] < 500]

if len(below_1000) > 0:
    print("⚠ Conditions with < 1000 cells after filtering:")
    print(below_1000)
else:
    print("✓ All conditions have ≥ 1000 cells")

if len(below_500g) > 0:
    print("\n⚠ Conditions with median genes < 500 after filtering:")
    print(below_500g)
else:
    print("✓ All conditions have median genes ≥ 500")

In [ ]:
# ── SAVE ──────────────────────────────────────────────────────────
adata.write_h5ad(OUTPUT)
print(f"Saved: {OUTPUT}")
print(adata)